# Comparative Machine Learning Analysis
## Phase 4: Multi-Algorithm Shear Strength Prediction

**Objective:** Predict Angle of Internal Friction (φ) and Undrained Cohesion (Cu) using three distinct ML approaches.

**Models:**
1. **Random Forest Regressor** - Ensemble baseline with feature importance
2. **XGBoost Regressor** - Gradient boosting for precision
3. **Support Vector Regressor (RBF)** - Kernel-based non-linear approach

**Evaluation:** 5-fold CV | Metrics: R², MAE, RMSE

---

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Models
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✓ Libraries imported successfully")
print(f"\nVersions:")
import sklearn
import xgboost as xgb
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  xgboost:      {xgb.__version__}")

---
## 1. Data Loading & Preparation

In [ ]:
# Load engineered dataset
df = pd.read_csv('/home/claude/geotechnical_engineered.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Variables:")
print(f"  Angle_Internal_Friction: {df['Angle_Internal_Friction'].notna().sum()} samples")
print(f"  Undrained_Cohesion:      {df['Undrained_Cohesion'].notna().sum()} samples")

In [ ]:
# Define feature set
FEATURE_CANDIDATES = [
    # Atterberg Limits
    'LL', 'PL', 'PI',
    
    # Physical Properties
    'Moisture_Content', 'Unit_Weight', 'Fines_Content',
    
    # In-situ Testing
    'SPT_N', 'CPT_qc_unified',
    
    # Engineered Features
    'Liquidity_Index', 'Consistency_Index', 'Activity_Ratio',
    
    # Depth (important for stress state)
    'Depth',
]

# Filter to available features
available_features = [f for f in FEATURE_CANDIDATES if f in df.columns]

print(f"\nAvailable Features ({len(available_features)}):")
for feat in available_features:
    non_null = df[feat].notna().sum()
    pct = 100 * non_null / len(df)
    print(f"  {feat:25s}: {non_null:5d} ({pct:5.1f}%)")

In [ ]:
# Prepare dataset for modeling
# Only keep records where at least one target is available
targets = ['Angle_Internal_Friction', 'Undrained_Cohesion']

df_modeling = df.copy()

# Filter: keep rows where at least one target exists
has_target = df_modeling[targets].notna().any(axis=1)
df_modeling = df_modeling[has_target].copy()

print(f"\n{'='*60}")
print("DATASET PREPARATION")
print(f"{'='*60}")
print(f"Original records:      {len(df)}")
print(f"Records with targets:  {len(df_modeling)}")
print(f"Features selected:     {len(available_features)}")
print(f"{'='*60}\n")

In [ ]:
# Handle missing values in features
# Strategy: Median imputation for numerical features
from sklearn.impute import SimpleImputer

X_raw = df_modeling[available_features].copy()
y_raw = df_modeling[targets].copy()

# Impute features
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index
)

print(f"Features after imputation: {X_imputed.shape}")
print(f"Targets: {y_raw.shape}")
print(f"\nMissing values in features: {X_imputed.isnull().sum().sum()}")
print(f"Missing values in targets:  {y_raw.isnull().sum().sum()}")

---
## 2. Train-Test Split & Scaling

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y_raw, test_size=0.2, random_state=42
)

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

# Feature scaling (critical for SVR)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Features scaled to zero mean and unit variance")

---
## 3. Model Training & Evaluation

### Strategy:
We'll train each model to predict both targets simultaneously using `MultiOutputRegressor`.

In [ ]:
# Evaluation helper functions
def evaluate_model(y_true, y_pred, model_name):
    """
    Calculate and display regression metrics.
    """
    results = {}
    
    for i, target in enumerate(targets):
        # Filter out NaN values in true labels
        mask = y_true.iloc[:, i].notna()
        y_t = y_true.iloc[:, i][mask]
        y_p = y_pred[mask, i]
        
        if len(y_t) > 0:
            r2 = r2_score(y_t, y_p)
            mae = mean_absolute_error(y_t, y_p)
            rmse = np.sqrt(mean_squared_error(y_t, y_p))
            
            results[target] = {
                'R²': r2,
                'MAE': mae,
                'RMSE': rmse,
                'n_samples': len(y_t)
            }
    
    return results

def print_results(results, model_name):
    """
    Pretty print evaluation results.
    """
    print(f"\n{'='*60}")
    print(f"{model_name.upper()} - PERFORMANCE METRICS")
    print(f"{'='*60}")
    
    for target, metrics in results.items():
        print(f"\n{target}:")
        print(f"  R² Score:  {metrics['R²']:7.4f}")
        print(f"  MAE:       {metrics['MAE']:7.4f}")
        print(f"  RMSE:      {metrics['RMSE']:7.4f}")
        print(f"  Samples:   {metrics['n_samples']:7d}")
    
    print(f"{'='*60}\n")

### 3.1 Random Forest Regressor

In [ ]:
print("Training Random Forest Regressor...\n")

# Initialize model
rf_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
)

# Train model
rf_model.fit(X_train_scaled, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test_scaled)

# Evaluate
rf_results = evaluate_model(y_test, y_pred_rf, "Random Forest")
print_results(rf_results, "Random Forest")

print("✓ Random Forest training complete")

### 3.2 XGBoost Regressor

In [ ]:
print("Training XGBoost Regressor...\n")

# Initialize model
xgb_model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
)

# Train model
xgb_model.fit(X_train_scaled, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test_scaled)

# Evaluate
xgb_results = evaluate_model(y_test, y_pred_xgb, "XGBoost")
print_results(xgb_results, "XGBoost")

print("✓ XGBoost training complete")

### 3.3 Support Vector Regressor (RBF Kernel)

In [ ]:
print("Training Support Vector Regressor...\n")

# Initialize model
svr_model = MultiOutputRegressor(
    SVR(
        kernel='rbf',
        C=100,
        gamma='scale',
        epsilon=0.1
    )
)

# Train model
svr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_svr = svr_model.predict(X_test_scaled)

# Evaluate
svr_results = evaluate_model(y_test, y_pred_svr, "SVR")
print_results(svr_results, "SVR (RBF)")

print("✓ SVR training complete")

---
## 4. Cross-Validation Analysis

5-fold CV for robust performance estimation.

In [ ]:
print(f"\n{'='*60}")
print("5-FOLD CROSS-VALIDATION")
print(f"{'='*60}\n")

# Prepare full dataset for CV
X_full = scaler.fit_transform(X_imputed)
y_full = y_raw.values

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Random Forest': MultiOutputRegressor(RandomForestRegressor(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
    )),
    'XGBoost': MultiOutputRegressor(XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1
    )),
    'SVR (RBF)': MultiOutputRegressor(SVR(kernel='rbf', C=100, gamma='scale'))
}

cv_results = {}

for model_name, model in models.items():
    print(f"Running CV for {model_name}...")
    
    # For each fold, train and evaluate
    fold_r2_scores = []
    
    for train_idx, val_idx in kfold.split(X_full):
        X_tr, X_val = X_full[train_idx], X_full[val_idx]
        y_tr, y_val = y_full[train_idx], y_full[val_idx]
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        
        # Calculate R² for each output
        r2_scores = []
        for i in range(y_val.shape[1]):
            mask = ~np.isnan(y_val[:, i])
            if mask.sum() > 0:
                r2 = r2_score(y_val[mask, i], y_pred[mask, i])
                r2_scores.append(r2)
        
        fold_r2_scores.append(np.mean(r2_scores))
    
    cv_results[model_name] = {
        'mean_r2': np.mean(fold_r2_scores),
        'std_r2': np.std(fold_r2_scores),
        'scores': fold_r2_scores
    }
    
    print(f"  Mean R² (CV): {cv_results[model_name]['mean_r2']:.4f} (+/- {cv_results[model_name]['std_r2']:.4f})\n")

print(f"{'='*60}\n")

---
## 5. Comparative Visualizations

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Angle of Internal Friction
target_idx = 0
r2_scores_phi = []
mae_scores_phi = []
model_names = ['Random Forest', 'XGBoost', 'SVR']

for model_name, results in zip(model_names, [rf_results, xgb_results, svr_results]):
    target_name = targets[target_idx]
    if target_name in results:
        r2_scores_phi.append(results[target_name]['R²'])
        mae_scores_phi.append(results[target_name]['MAE'])

x_pos = np.arange(len(model_names))
axes[0].bar(x_pos, r2_scores_phi, color=['steelblue', 'coral', 'seagreen'], alpha=0.8, edgecolor='black')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('Model Comparison: Angle of Internal Friction (φ)', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(r2_scores_phi):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# Undrained Cohesion
target_idx = 1
r2_scores_cu = []
mae_scores_cu = []

for model_name, results in zip(model_names, [rf_results, xgb_results, svr_results]):
    target_name = targets[target_idx]
    if target_name in results:
        r2_scores_cu.append(results[target_name]['R²'])
        mae_scores_cu.append(results[target_name]['MAE'])

axes[1].bar(x_pos, r2_scores_cu, color=['steelblue', 'coral', 'seagreen'], alpha=0.8, edgecolor='black')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('Model Comparison: Undrained Cohesion (Cu)', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(r2_scores_cu):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/home/claude/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Model comparison chart saved")

### Actual vs. Predicted Scatter Plots

In [ ]:
def plot_actual_vs_predicted(y_true, y_pred_dict, target_idx, target_name):
    """
    Create actual vs predicted scatter plots for all models.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, (model_name, y_pred) in enumerate(y_pred_dict.items()):
        # Filter valid predictions
        mask = y_true.iloc[:, target_idx].notna()
        y_t = y_true.iloc[:, target_idx][mask]
        y_p = y_pred[mask, target_idx]
        
        if len(y_t) > 0:
            # Scatter plot
            axes[idx].scatter(y_t, y_p, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
            
            # Perfect prediction line
            min_val = min(y_t.min(), y_p.min())
            max_val = max(y_t.max(), y_p.max())
            axes[idx].plot([min_val, max_val], [min_val, max_val], 
                          'r--', linewidth=2, label='Perfect Prediction')
            
            # Calculate R²
            r2 = r2_score(y_t, y_p)
            
            axes[idx].set_xlabel(f'Actual {target_name}', fontsize=12)
            axes[idx].set_ylabel(f'Predicted {target_name}', fontsize=12)
            axes[idx].set_title(f'{model_name}\n(R² = {r2:.3f})', 
                               fontsize=13, fontweight='bold')
            axes[idx].legend()
            axes[idx].grid(alpha=0.3)
    
    plt.tight_layout()
    filename = f'/home/claude/actual_vs_predicted_{target_name.replace(" ", "_").lower()}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}\n")

# Generate plots for both targets
predictions = {
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'SVR': y_pred_svr
}

for idx, target in enumerate(targets):
    print(f"Generating scatter plots for {target}...")
    plot_actual_vs_predicted(y_test, predictions, idx, target)

---
## 6. Feature Importance Analysis

In [ ]:
print(f"\n{'='*60}")
print("FEATURE IMPORTANCE ANALYSIS")
print(f"{'='*60}\n")

# Random Forest Feature Importance
print("Random Forest - Feature Importance:\n")

# Extract importances from each output's estimator
rf_importances = []
for estimator in rf_model.estimators_:
    rf_importances.append(estimator.feature_importances_)

rf_importance_df = pd.DataFrame({
    'Feature': available_features,
    'Importance_Target_0': rf_importances[0],
    'Importance_Target_1': rf_importances[1] if len(rf_importances) > 1 else np.zeros(len(available_features)),
    'Mean_Importance': np.mean(rf_importances, axis=0)
}).sort_values('Mean_Importance', ascending=False)

print(rf_importance_df.head(10))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
top_features = rf_importance_df.head(10)
ax.barh(top_features['Feature'], top_features['Mean_Importance'], color='steelblue', edgecolor='black')
ax.set_xlabel('Mean Importance', fontsize=12)
ax.set_title('Random Forest - Top 10 Feature Importances', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/home/claude/feature_importance_rf.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved")

In [ ]:
# XGBoost Feature Importance
print("\nXGBoost - Feature Importance:\n")

xgb_importances = []
for estimator in xgb_model.estimators_:
    xgb_importances.append(estimator.feature_importances_)

xgb_importance_df = pd.DataFrame({
    'Feature': available_features,
    'Importance_Target_0': xgb_importances[0],
    'Importance_Target_1': xgb_importances[1] if len(xgb_importances) > 1 else np.zeros(len(available_features)),
    'Mean_Importance': np.mean(xgb_importances, axis=0)
}).sort_values('Mean_Importance', ascending=False)

print(xgb_importance_df.head(10))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
top_features = xgb_importance_df.head(10)
ax.barh(top_features['Feature'], top_features['Mean_Importance'], color='coral', edgecolor='black')
ax.set_xlabel('Mean Importance', fontsize=12)
ax.set_title('XGBoost - Top 10 Feature Importances', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/home/claude/feature_importance_xgb.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ XGBoost feature importance plot saved")

---
## 7. Model Persistence

In [ ]:
import pickle

# Save all trained models
models_to_save = {
    'random_forest': rf_model,
    'xgboost': xgb_model,
    'svr': svr_model,
    'scaler': scaler
}

for model_name, model_obj in models_to_save.items():
    filepath = f'/home/claude/model_{model_name}.pkl'
    with open(filepath, 'wb') as f:
        pickle.dump(model_obj, f)
    print(f"✓ Saved: {filepath}")

# Save feature names
with open('/home/claude/feature_names.pkl', 'wb') as f:
    pickle.dump(available_features, f)

print("\n✓ All models and metadata saved for deployment")

---
## 8. Executive Summary

In [ ]:
print(f"\n{'='*70}")
print("GEOTECHNICAL SHEAR STRENGTH PREDICTION - EXECUTIVE SUMMARY")
print(f"{'='*70}\n")

print("DATASET STATISTICS:")
print(f"  Total records processed:    {len(df_modeling)}")
print(f"  Features engineered:        {len(available_features)}")
print(f"  Training samples:           {len(X_train)}")
print(f"  Test samples:               {len(X_test)}")

print("\nMODEL PERFORMANCE (Test Set R²):")
print("\n  Angle of Internal Friction (φ):")
for model_name, results in zip(['Random Forest', 'XGBoost', 'SVR'], 
                                [rf_results, xgb_results, svr_results]):
    if targets[0] in results:
        r2 = results[targets[0]]['R²']
        mae = results[targets[0]]['MAE']
        print(f"    {model_name:20s}: R² = {r2:.4f},  MAE = {mae:.2f}°")

print("\n  Undrained Cohesion (Cu):")
for model_name, results in zip(['Random Forest', 'XGBoost', 'SVR'], 
                                [rf_results, xgb_results, svr_results]):
    if targets[1] in results:
        r2 = results[targets[1]]['R²']
        mae = results[targets[1]]['MAE']
        print(f"    {model_name:20s}: R² = {r2:.4f},  MAE = {mae:.2f} kPa")

print("\nCROSS-VALIDATION RESULTS (5-Fold):")
for model_name, cv_result in cv_results.items():
    print(f"  {model_name:20s}: {cv_result['mean_r2']:.4f} (+/- {cv_result['std_r2']:.4f})")

print("\nTOP 5 PREDICTIVE FEATURES (Random Forest):")
for idx, row in rf_importance_df.head(5).iterrows():
    print(f"  {row['Feature']:25s}: {row['Mean_Importance']:.4f}")

print(f"\n{'='*70}")
print("RECOMMENDATION:")

# Determine best model based on average R² across targets
avg_r2 = {}
for model_name, results in zip(['Random Forest', 'XGBoost', 'SVR'], 
                                [rf_results, xgb_results, svr_results]):
    r2_scores = [results[t]['R²'] for t in targets if t in results]
    avg_r2[model_name] = np.mean(r2_scores)

best_model = max(avg_r2, key=avg_r2.get)

print(f"  Deploy {best_model} for production (highest avg R²: {avg_r2[best_model]:.4f})")
print(f"  All models saved to /home/claude/model_*.pkl for ensemble deployment")
print(f"{'='*70}\n")

---
## Summary

**Phase 4 Complete:**
1. ✓ Three models trained and evaluated (RF, XGBoost, SVR)
2. ✓ 5-fold cross-validation performed
3. ✓ Comprehensive performance metrics (R², MAE, RMSE)
4. ✓ Actual vs. Predicted visualizations
5. ✓ Feature importance rankings
6. ✓ Models persisted for deployment

**Ready for Production Deployment**